# 第 29 课：WebSocket 流式服务——会话状态、cache 与 PGS 事件

每个 WebSocket 连接拥有独立的 feature buffer、encoder cache、CTC previous token 和 transcript。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 量化与部署 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 28 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | WebSocket session、backpressure、PGS event |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：WebSocket session、backpressure、PGS event。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [1]:
from pathlib import Path
import time
import sys
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT=find_root(); ARTIFACTS=ROOT/"artifacts";ARTIFACTS.mkdir(exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

from fastapi.testclient import TestClient
from deployment.app import app
from deployment.model import FEATURE_DIM,CHUNK_FRAMES
client=TestClient(app);rng=np.random.default_rng(6)

项目根目录: G:\learn_asr


G:\learn_asr\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. 半块不会立即推理

服务端模型 contract 是 8 帧。先发送 4 帧，服务端缓存；再发送 4 帧才产生一个结果事件。

In [2]:
with client.websocket_connect("/stream") as ws:
    ws.send_json({"features":rng.normal(size=(4,FEATURE_DIM)).tolist()})
    ws.send_json({"features":rng.normal(size=(4,FEATURE_DIM)).tolist()})
    first=ws.receive_json();print("first:",first)
    ws.send_json({"features":rng.normal(size=(CHUNK_FRAMES,FEATURE_DIM)).tolist()})
    second=ws.receive_json();print("second:",second)
    ws.send_json({"eof":True})
    final=ws.receive_json();print("final:",final)

first: {'sn': 1, 'pgs': 'apd', 'text': '7832650', 'full': '7832650', 'ls': False}
second: {'sn': 2, 'pgs': 'apd', 'text': '20676535', 'full': '783265020676535', 'ls': False}
final: {'sn': 3, 'pgs': 'apd', 'text': '', 'full': '783265020676535', 'ls': True}


## 2. 四层状态不要混在一起

```text
前端状态：未成帧的 PCM/feature 尾部
编码器状态：卷积/注意力/RNN cache
解码器状态：CTC prefix、WFST active state、LM history
协议状态：sn、PGS slices、stable/final 标记
```

连接断开时要明确哪些状态销毁，哪些能通过 session id 恢复。

## 3. Backpressure

如果客户端发送速度超过服务端处理速度，队列会无限增长。常见策略：限制每连接缓冲、暂停读取、拒绝新连接、降低 beam、分配独立 worker；不能悄悄丢掉中间音频。

## 4. 当前教学服务为什么接收 feature 而不是 PCM

这是为了单独验证模型 serving、cache 和 PGS。生产系统通常接收二进制 PCM/Opus，并在服务端执行有状态重采样、分帧、Log-Mel 和 CMVN；也可以把前端部署到设备侧，但必须固定特征规范。

## 本课测试

1. 每条 WebSocket 连接应共享还是隔离 decoder state？
2. 为什么不能在 chunk 结束时清空 previous token？
3. EOF 到达时应做哪些 flush？
4. backpressure 为什么影响稳定性？
5. PGS `rpl` 事件在客户端应如何处理？

<details><summary>展开参考答案</summary>

1. 隔离。2. 跨 chunk 重复折叠依赖它。3. 处理尾部帧策略、完成 decoder/endpoint、发送 final 并释放状态。4. 队列增长会增加延迟和内存，最终雪崩。5. 按 rg 替换历史 slice，而不是简单追加。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 29 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `WebSocket session`、`backpressure`、`PGS event`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**客户端发送速度高于推理速度**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**测试两连接状态隔离、EOF 与断线**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接模型 cache、decoder state 和协议 state**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：WebSocket session、backpressure、PGS event。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 WebSocket session、backpressure、PGS event。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
